# 서울 열린데이터 광장 사이트에서 공공데이터 목록 수집하기

In [1]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
from tqdm import tqdm

## 데이터 수집 목록
- 1페이지의 데이터: 타이틀, 보건, 수정일자, 제공부서

In [2]:
U_A = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"}

In [127]:
url = 'https://data.seoul.go.kr/dataList/datasetList.do#'
res = requests.get(url, headers=U_A)
res

<Response [200]>

In [5]:
soup = bs(res.text, 'lxml')
soup

<!DOCTYPE html>
<html lang="ko">
<head>
<!-- Global site tag (gtag.js) Google Analytics -->
<script async="" src="//www.googletagmanager.com/gtag/js?id=G-0T3XG23CN7"></script>
<script>
		window.dataLayer = window.dataLayer || [];
		function gtag(){dataLayer.push(arguments);}
		gtag('js', new Date());
		gtag('config', 'G-0T3XG23CN7');
	</script>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="" name="title"/>
<meta content="" name="keywords"/>
<meta content="" name="description"/>
<meta content="width=device-width" name="viewport"/>
<!-- meta name="viewport" content="width=device-width,initial-scale=1.0,minimum-scale=1.0,maximum-scale=1.0,user-scalable=no"/--><!-- ios때문에 이렇게 한다고하는데..w3s의 권장은 윗줄 -->
<meta content="website" property="og:type"/>
<meta content="" property="og:title"/>
<meta content="" property="og:description"/>
<meta content="https://data.seoul.go.kr:8580/resources/img/common/logo.pn

### 1) 타이틀

In [104]:
title = soup.select('a.goView strong')
print(len(title))

for i in title:
    print(i.text)

10
서울시 위생처리업 현황 
서울시 세척제 제조업 현황 
서울시 중구 위생처리업 공중위생업소 현황 
서울시 노동조합 현황 
서울시 기타 위생용품 제조업 현황 
서울시 식품위생업소 현황 
서울시 식품위생업소 (층) 현황 
상상대로 서울 자유제안 정보 
서울시 대부업체 정보 
서울글로벌센터 접수유형 및 상담유형별(월) 상담실적 


### 2) 분야

In [108]:
ctg = soup.select('dt em:nth-child(2)')
            # 카테고리마다 이름이 달랐는데,, 그걸 통합해줄 수 있는 상위로 가서 공통점으로 묶어주기!!
print(len(ctg))

for i in ctg:
    print(i.text)

10
[보건]
[보건]
[보건]
[산업/경제]
[보건]
[보건]
[보건]
[일반행정]
[산업/경제]
[복지]


In [107]:
# 조금 더 안전하게 가려면~~~ 내가 원하는 데이터가 있는 전체!를 선택해서 그 안으로 파고들기~
ctg = soup.select('div.list-statistics em:nth-child(2)')
ctg

[<em class="stat-class-1">[보건]</em>,
 <em class="stat-class-1">[보건]</em>,
 <em class="stat-class-1">[보건]</em>,
 <em class="stat-class-4">[산업/경제]</em>,
 <em class="stat-class-1">[보건]</em>,
 <em class="stat-class-1">[보건]</em>,
 <em class="stat-class-1">[보건]</em>,
 <em class="stat-class-2">[일반행정]</em>,
 <em class="stat-class-4">[산업/경제]</em>,
 <em class="stat-class-5">[복지]</em>]

### 3) 날짜

In [118]:
date = soup.select('dd.list-statistics-info2 > span:first-child')
print(len(date))
date[1].text.strip()

date_list = []
for i in date:
    print(i.text.strip()[-10:]) # 뒤에서 10자리 슬라이싱
    # date_list.append(i.text.replace('\t', '').split(':')[-1].strip())

# date_list

10
2026-01-26
2026-01-26
2026-01-26
2026-01-26
2026-01-26
2026-01-26
2026-01-26
2026-01-26
2026-01-26
2026-01-26


### 4) 부서

In [112]:
dpt = soup.select('dd.list-statistics-info2 > span:nth-child(3)')
dpt[0].text.strip()

dpt_list = []
for i in dpt:
    dpt_list.append(i.text.replace('\t', '').split(':')[-1].strip())

dpt_list

['디지털도시국 데이터전략과',
 '디지털도시국 데이터전략과',
 '디지털도시국 정보시스템과',
 '민생노동국 노동정책과',
 '시민건강국 감염병관리과',
 '디지털도시국 데이터전략과',
 '디지털도시국 데이터전략과',
 '홍보기획관 콘텐츠담당관',
 '노동공정상생정책관 공정경제담당관',
 '경제실 경제일자리기획관 금융투자과']

## 2page 정보 수집
- POST 방식

In [128]:
payload = {'pageIndex': 2}

# post: POST 방식으로 요청
# data: 폼 데이터를 설정하는 매개변수로 웹 사이트 접근 시 제출해야하는 정보를 포함시킴
res_seoul = requests.post(url, data=payload)
res_seoul

<Response [200]>

In [129]:
soup = bs(res_seoul.text, 'lxml')

title = soup.select('a.goView strong')
print(len(title))
for i in title:
    print(i.text)

10
서울글로벌센터 상담유형별(월) 상담실적 
서울글로벌센터 접수유형별 언어별 상담실적 
서울글로벌센터 상담유형별(분기) 상담실적 
서울시 집수리닷컴 공지사항 
서울시 집수리 시공업체 정보 
집수리 아카데미 수료자 활동내역 
서울글로벌센터 언어별(월) 상담실적 
서울시 기타조형물 현황 
집수리 성공기 
서울은미술관 현황 


## 1~5page의 총 50개의 정보를 수집 후 데이터 프레임으로 출력해보세요

In [152]:
url = 'https://data.seoul.go.kr/dataList/datasetList.do'

title_list, ctg_list, date_list, dpt_list = [], [], [], []

for i in tqdm(range(1, 6)):
    payload = {'pageIndex': {i}}
    res_seoul = requests.post(url, data=payload, headers=U_A)
    soup = bs(res_seoul.text, 'lxml')

    title = soup.select('a.goView strong')
    ctg = soup.select('dt em:nth-child(2)')
    date = soup.select('dd.list-statistics-info2 > span:nth-child(1)')
    dpt = soup.select('dd.list-statistics-info2 > span:nth-child(3)')

    for i in range(len(title)):
        title_list.append(title[i].text.strip())
        ctg_list.append(ctg[i].text.strip())
        date_list.append(date[i].text.strip()[-10:])             
        dpt_list.append(dpt[i].text.strip()[19:])

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:06<00:00,  1.37s/it]


In [153]:
data_dic = {'제목':title_list, '카테고리':ctg_list, '날짜':date_list, '부서':dpt_list}
df = pd.DataFrame(data_dic, index=range(1, len(dpt_list)+1))
df

,제목,카테고리,날짜,부서
1,서울시 위생처리업 현황,[보건],2026-01-26,디지털도시국 데이터전략과
2,서울시 세척제 제조업 현황,[보건],2026-01-26,디지털도시국 데이터전략과
3,서울시 노동조합 현황,[산업/경제],2026-01-26,민생노동국 노동정책과
4,서울시 기타 위생용품 제조업 현황,[보건],2026-01-26,시민건강국 감염병관리과
5,서울시 중구 위생처리업 공중위생업소 현황,[보건],2026-01-26,디지털도시국 정보시스템과
6,서울시 식품위생업소 현황,[보건],2026-01-26,디지털도시국 데이터전략과
7,상상대로 서울 자유제안 정보,[일반행정],2026-01-26,홍보기획관 콘텐츠담당관
8,서울시 식품위생업소 (층) 현황,[보건],2026-01-26,디지털도시국 데이터전략과
9,서울시 대부업체 정보,[산업/경제],2026-01-26,노동공정상생정책관 공정경제담당관
10,서울글로벌센터 접수유형 및 상담유형별(월) 상담실적,[복지],2026-01-26,경제실 경제일자리기획관 금융투자과
